In [1]:
import os 
import pandas as pd 
import json

In [3]:
edamam_dir = 'edamam_results'
edamam_json_files = [f for f in os.listdir(edamam_dir) if f.endswith('.json')]

In [20]:
import os
import json

processed_data = []

for edamam_json in edamam_json_files:
    path = os.path.join(edamam_dir, edamam_json)
    with open(path, 'r') as file:
        data = json.load(file)
        result = {
            'search_text': data['text'],
            'parsed': None,
            'hints': []
        }
        
        # Extract parsed data if it exists
        if 'parsed' in data and data['parsed']:
            food = data['parsed'][0]['food']
            result['parsed'] = {
                'foodId': food.get('foodId'),
                'label': food.get('label'),
                'knownAs': food.get('knownAs'),
                'category': food.get('category')
            }
        
        # Extract relevant data from hints
        if 'hints' in data:
            for hint in data['hints']:
                if 'food' in hint:
                    food = hint['food']
                    result['hints'].append({
                        'foodId': food.get('foodId'),
                        'label': food.get('label'),
                        'knownAs': food.get('knownAs'),
                        'category': food.get('category')
                    })
        
        processed_data.append(result)


In [22]:
processed_data[0]

{'search_text': 'chocolate cookie',
 'parsed': {'foodId': 'food_bx579krbnzvu5jb0h2mn0bv9doxe',
  'label': 'Chocolate Cookies',
  'knownAs': 'chocolate wafer cookies',
  'category': 'Generic foods'},
 'hints': [{'foodId': 'food_bx579krbnzvu5jb0h2mn0bv9doxe',
   'label': 'Chocolate Cookies',
   'knownAs': 'chocolate wafer cookies',
   'category': 'Generic foods'},
  {'foodId': 'food_bx579krbnzvu5jb0h2mn0bv9doxe',
   'label': 'Chocolate Cookie Wafer',
   'knownAs': 'chocolate wafer cookies',
   'category': 'Generic foods'},
  {'foodId': 'food_bx579krbnzvu5jb0h2mn0bv9doxe',
   'label': 'Chocolate Wafer Cookie',
   'knownAs': 'chocolate wafer cookies',
   'category': 'Generic foods'},
  {'foodId': 'food_avj86tfbfmsho4a79sd2rb4fwovc',
   'label': 'Chocolate Chocolate Cookies',
   'knownAs': 'Chocolate Chocolate Cookies',
   'category': 'Generic meals'},
  {'foodId': 'food_adjtj43btzf602bpsc668bgbtubg',
   'label': 'Chocolate Cookies',
   'knownAs': 'Chocolate Cookies',
   'category': 'Generi

In [23]:
import os
import pandas as pd
import json
import requests
import time
from tqdm.auto import tqdm
from google import genai
from dotenv import load_dotenv
import pickle
import random

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Initialize Google Gemini client
client = genai.Client(api_key=GOOGLE_API_KEY)


In [ ]:
import os
import json
from google import genai
from dotenv import load_dotenv
import re

# Load environment variables
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Initialize Google Gemini client
client = genai.Client(api_key=GOOGLE_API_KEY)

# Create output directory if it doesn't exist
gemini_dir = 'gemini_edamam'
os.makedirs(gemini_dir, exist_ok=True)

def sanitize_filename(filename):
   # Replace spaces with underscores and remove invalid characters
   return re.sub(r'[^\w\-_]', '', filename.replace(' ', '_'))

def find_most_relevant_food(search_text, parsed_data, hints_data):
   """
   Use Gemini to determine the most relevant food item from Edamam results
   """
   food_data_str = json.dumps({
       "search_text": search_text,
       "parsed": parsed_data,
       "hints": hints_data
   }, indent=2)
   
   prompt = f"""
   FOOD SEARCH: "{search_text}"
   
   AVAILABLE FOOD DATABASE RESULTS:
   {food_data_str}
   
   Based on the search text, determine which food item is most relevant.
   Return ONLY the foodId of the most relevant item in this format:
   {{
     "foodId": "selected_food_id_here"
   }}
   """
   
   try:
       response = client.models.generate_content(
           model="gemini-2.0-flash",
           contents=prompt,
           config={
               'response_mime_type': 'application/json',
               'temperature': 0.0
           }
       )
       
       result = json.loads(response.text)
       return result.get('foodId')
   
   except Exception as e:
       print(f"Error determining relevant food for '{search_text}': {e}")
       if parsed_data:
           return parsed_data.get('foodId')
       elif hints_data:
           return hints_data[0].get('foodId')
       return None

# Process each JSON file
for edamam_json in edamam_json_files:
   path = os.path.join(edamam_dir, edamam_json)
   with open(path, 'r') as file:
       data = json.load(file)
       search_text = data['text']
       
       # Create filename for Gemini results
       output_filename = f"{sanitize_filename(search_text)}.json"
       output_path = os.path.join(gemini_dir, output_filename)
       
       # Check if we already have processed this search
       if os.path.exists(output_path):
           print(f"Already processed: {search_text}")
           continue
           
       # Process data
       result = {
           'search_text': search_text,
           'parsed': None,
           'hints': []
       }
       
       # Extract parsed data if it exists
       if 'parsed' in data and data['parsed']:
           food = data['parsed'][0]['food']
           result['parsed'] = {
               'foodId': food.get('foodId'),
               'label': food.get('label'),
               'knownAs': food.get('knownAs'),
               'category': food.get('category')
           }
       
       # Extract relevant data from hints
       if 'hints' in data:
           for hint in data['hints']:
               if 'food' in hint:
                   food = hint['food']
                   result['hints'].append({
                       'foodId': food.get('foodId'),
                       'label': food.get('label'),
                       'knownAs': food.get('knownAs'),
                       'category': food.get('category')
                   })
       
       # Get most relevant food ID
       most_relevant_id = find_most_relevant_food(
           search_text, 
           result['parsed'], 
           result['hints']
       )
       
       # Add most relevant ID to result
       result['most_relevant_id'] = most_relevant_id
       
       # Save to file
       with open(output_path, 'w') as outfile:
           json.dump(result, outfile, indent=2)
           
       print(f"Processed '{search_text}', saved to {output_filename}")
       break

For 'chocolate cookie', most relevant foodId: food_bx579krbnzvu5jb0h2mn0bv9doxe


In [29]:
import os
import json
import time
import re
from tqdm import tqdm
from google import genai
from dotenv import load_dotenv
import random

# Setup
load_dotenv()
gemini_dir = 'gemini_edamam'
os.makedirs(gemini_dir, exist_ok=True)
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

def sanitize_filename(filename):
    return re.sub(r'[^\w\-_]', '', filename.replace(' ', '_'))

def find_most_relevant_food(search_text, parsed_data, hints_data, max_retries=3):
    prompt = f"""
    FOOD SEARCH: "{search_text}"
    
    AVAILABLE FOOD DATABASE RESULTS:
    {json.dumps({"search_text": search_text, "parsed": parsed_data, "hints": hints_data}, indent=2)}
    
    Based on the search text, determine which food item is most relevant.
    Return ONLY the foodId of the most relevant item in this format:
    {{
      "foodId": "selected_food_id_here"
    }}
    """
    
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-2.0-flash",
                contents=prompt,
                config={'response_mime_type': 'application/json', 'temperature': 0.0}
            )
            return json.loads(response.text).get('foodId')
        except Exception as e:
            wait_time = (2 ** attempt) + random.uniform(0, 1)
            print(f"Attempt {attempt+1} failed: {e}. Retrying in {wait_time:.2f}s...")
            time.sleep(wait_time)
    
    # Fallback
    print(f"All retries failed for '{search_text}'")
    return parsed_data.get('foodId') if parsed_data else hints_data[0].get('foodId') if hints_data else None

# Process each JSON file
for edamam_json in tqdm(edamam_json_files, desc="Processing food items"):
    path = os.path.join(edamam_dir, edamam_json)
    with open(path, 'r') as file:
        data = json.load(file)
        search_text = data['text']
        
        output_path = os.path.join(gemini_dir, f"{sanitize_filename(search_text)}.json")
        
        # Skip if already processed
        if os.path.exists(output_path):
            continue
        
        # Extract data
        result = {'search_text': search_text, 'parsed': None, 'hints': []}
        
        if 'parsed' in data and data['parsed']:
            food = data['parsed'][0]['food']
            result['parsed'] = {k: food.get(k) for k in ['foodId', 'label', 'knownAs', 'category']}
        
        if 'hints' in data:
            result['hints'] = [
                {k: food.get(k) for k in ['foodId', 'label', 'knownAs', 'category']}
                for hint in data['hints'] if 'food' in hint 
                for food in [hint['food']]
            ]
        
        # Get most relevant food ID and save
        result['most_relevant_id'] = find_most_relevant_food(
            search_text, result['parsed'], result['hints']
        )
        
        with open(output_path, 'w') as outfile:
            json.dump(result, outfile, indent=2)
        
        time.sleep(0.5)


Processing food items: 100%|██████████| 2663/2663 [44:43<00:00,  1.01s/it]


In [ ]:

# Summary
processed_count = len(os.listdir(gemini_dir))
print(f"Processing complete: {processed_count}/{len(edamam_json_files)} items processed")
print(f"Results saved to {gemini_dir}/ directory")

def lookup_food_by_id(foodId, all_results):
    for result in all_results:
        if result['parsed'] and result['parsed'].get('foodId') == foodId:
            return result['parsed']
        for hint in result['hints']:
            if hint.get('foodId') == foodId:
                return hint
    return None

def analyze_results():
    all_results = [json.load(open(os.path.join(gemini_dir, f), 'r')) 
                  for f in os.listdir(gemini_dir)]
    
    parsed_chosen = sum(1 for r in all_results if r['most_relevant_id'] and 
                        r['parsed'] and r['parsed'].get('foodId') == r['most_relevant_id'])
    hint_chosen = sum(1 for r in all_results if r['most_relevant_id'] and
                     (not r['parsed'] or r['parsed'].get('foodId') != r['most_relevant_id']))
    
    print(f"\nAnalysis: {len(all_results)} total items")
    print(f"Gemini chose parsed: {parsed_chosen} ({parsed_chosen/len(all_results)*100:.1f}%)")
    print(f"Gemini chose hint: {hint_chosen} ({hint_chosen/len(all_results)*100:.1f}%)")

# Uncomment to run analysis
# analyze_results()

In [28]:
import os
import json
from collections import Counter

def analyze_edamam_results():
    gemini_dir = 'gemini_edamam'
    
    stats = {
        'total': 0,
        'matches_parsed': 0,
        'matches_first_hint': 0,
        'other_hint': 0,
        'no_relevant_id': 0
    }
    
    interesting_cases = []
    
    # Analyze each file
    for filename in os.listdir(gemini_dir):
        if not filename.endswith('.json'):
            continue
            
        with open(os.path.join(gemini_dir, filename), 'r') as f:
            data = json.load(f)
            
        stats['total'] += 1
        
        if not data.get('most_relevant_id'):
            stats['no_relevant_id'] += 1
        elif data.get('parsed') and data['parsed'].get('foodId') == data['most_relevant_id']:
            stats['matches_parsed'] += 1
        elif data.get('hints') and data['hints'] and data['hints'][0].get('foodId') == data['most_relevant_id']:
            stats['matches_first_hint'] += 1
        else:
            stats['other_hint'] += 1
            
            # Save interesting case for further analysis
            chosen_hint = next((h for h in data.get('hints', []) if h.get('foodId') == data['most_relevant_id']), None)
            interesting_cases.append({
                'search': data['search_text'],
                'chosen': chosen_hint.get('label') if chosen_hint else 'Unknown',
                'chosen_id': data['most_relevant_id'],
                'parsed': data.get('parsed', {}).get('label') if data.get('parsed') else None,
                'first_hint': data.get('hints', [{}])[0].get('label') if data.get('hints') else None
            })
    
    # Print summary
    print(f"Total files analyzed: {stats['total']}")
    print(f"Matches parsed: {stats['matches_parsed']} ({stats['matches_parsed']/stats['total']*100:.1f}%)")
    print(f"Matches first hint: {stats['matches_first_hint']} ({stats['matches_first_hint']/stats['total']*100:.1f}%)")
    print(f"Matches other hint: {stats['other_hint']} ({stats['other_hint']/stats['total']*100:.1f}%)")
    print(f"No relevant ID: {stats['no_relevant_id']} ({stats['no_relevant_id']/stats['total']*100:.1f}%)")
    
    # Show examples of interesting cases
    if interesting_cases:
        print("\nSome examples where a different hint was chosen:")
        for i, case in enumerate(interesting_cases[:5]):  # Show up to 5 examples
            print(f"\n{i+1}. Search: '{case['search']}'")
            print(f"   Chosen: {case['chosen']} ({case['chosen_id']})")
            print(f"   Parsed: {case['parsed']}")
            print(f"   First hint: {case['first_hint']}")
    
    return stats, interesting_cases

# Run the analysis
stats, cases = analyze_edamam_results()

Total files analyzed: 102
Matches parsed: 52 (51.0%)
Matches first hint: 25 (24.5%)
Matches other hint: 24 (23.5%)
No relevant ID: 1 (1.0%)

Some examples where a different hint was chosen:

1. Search: 'green apple amino energy'
   Chosen: Optimum Nutrition Essential Amino Energy Green Apple 9.5 Oz (food_ag8ur2qa28dhkrbxcngxjbr5x1n2)
   Parsed: Green Apple
   First hint: Green Apple

2. Search: 'lipton'
   Chosen: Lipton Iced Tea (food_ba277cabrre4v0avejmf8apnqj6t)
   Parsed: None
   First hint: Lipton Berry Iced Tea With a Splash of Juice, Berry

3. Search: 'spring roll'
   Chosen: Spring Rolls (food_b8vpe0vb5s6trubk0452las0tcww)
   Parsed: Roll
   First hint: Roll

4. Search: 'girl scout cookie'
   Chosen: Girl Scouts, Cookies, Caramel Delites (food_a2goi8ha7rent1ar15avrb670njq)
   Parsed: Cookie
   First hint: Cookie

5. Search: 'honeycrisp'
   Chosen: Honeycrisp Apples (food_azmi0skbnwd9ahbxmjvb1b1rv7tb)
   Parsed: None
   First hint: Mini Scones With Cinnamon, Honeycrisp Apple
